In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
batch_size = 32
image_size = (128, 128)

In [3]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.3,
    shear_range=0.3,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3]
)

val_datagen = ImageDataGenerator(rescale=1./255)  

In [4]:
train_data = train_datagen.flow_from_directory(
    "split_data/train",
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical'
)

val_data = val_datagen.flow_from_directory(
    "split_data/val",
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 13726 images belonging to 5 classes.
Found 2722 images belonging to 5 classes.


In [5]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)



94765736/94765736 [==============================] - 9s 0us/step


In [6]:
for layer in base_model.layers:
    layer.trainable = False

In [7]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)

x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)

output = layers.Dense(train_data.num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

In [8]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [9]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [10]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=[early_stop]
) 

Epoch 1/20


429/429 [==============================] - 727s 2s/step - loss: 1.4240 - accuracy: 0.3794 - val_loss: 1.3572 - val_accuracy: 0.4482
Epoch 2/20
429/429 [==============================] - 683s 2s/step - loss: 1.3069 - accuracy: 0.4497 - val_loss: 1.1612 - val_accuracy: 0.5250
Epoch 3/20
429/429 [==============================] - 539s 1s/step - loss: 1.2469 - accuracy: 0.4870 - val_loss: 1.0968 - val_accuracy: 0.5658
Epoch 4/20
429/429 [==============================] - 505s 1s/step - loss: 1.2107 - accuracy: 0.5065 - val_loss: 1.0544 - val_accuracy: 0.6021
Epoch 5/20
429/429 [==============================] - 526s 1s/step - loss: 1.1699 - accuracy: 0.5262 - val_loss: 1.0326 - val_accuracy: 0.6113
Epoch 6/20
429/429 [==============================] - 503s 1s/step - loss: 1.1485 - accuracy: 0.5352 - val_loss: 0.9910 - val_accuracy: 0.6205
Epoch 7/20
429/429 [==============================] - 537s 1s/step - loss: 1.1163 - accuracy: 0.5535 - val_loss: 0.9703 - val_accuracy: 0.62

In [11]:
model.save('ResNet50.h5')

C:\Users\Asus\AppData\Roaming\Python\Python310\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [13]:
y_true = val_data.classes

y_pred_probs = model.predict(val_data)
y_pred = np.argmax(y_pred_probs, axis=1)

86/86 [==============================] - 84s 961ms/step


In [14]:
class_names = list(val_data.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_names))

              precision    recall  f1-score   support

       Boron       0.72      0.66      0.69       366
     Healthy       0.74      0.84      0.78       463
      Kalium       0.54      0.76      0.63       777
          Mg       0.66      0.53      0.59       734
    nitrogen       0.91      0.41      0.57       382

    accuracy                           0.65      2722
   macro avg       0.71      0.64      0.65      2722
weighted avg       0.68      0.65      0.64      2722



In [15]:
cm = confusion_matrix(y_true, y_pred)

print(cm)

[[243  15  96  11   1]
 [  0 388  48  20   7]
 [ 65  35 590  82   5]
 [ 25  79 236 391   3]
 [  5  10 125  85 157]]
